In [ ]:

# ============================================================
# BLOCK 0: SHARED SETUP — run this first in every session
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import torch

print("GPU available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

# Load + filter restaurant/English subset
df = pd.read_parquet("/content/drive/MyDrive/MABSA_Shared_Project/data/master_dataset.parquet")
df["domain"] = df["domain"].str.strip().str.lower()
df["language"] = df["language"].str.strip().str.lower()
restaurant_en = df[(df.domain == "restaurant") & (df.language == "en")].copy()
print("restaurant_en shape:", restaurant_en.shape)

# Label maps
sentiments = ["positive", "negative", "neutral"]
restaurant_categories = sorted(restaurant_en["category"].dropna().unique())

tasd_labels = ["O"]
for cat in restaurant_categories:
    for sent in sentiments:
        tasd_labels.append(f"B-{cat}-{sent}")
        tasd_labels.append(f"I-{cat}-{sent}")
tasd_label2id = {l: i for i, l in enumerate(tasd_labels)}

uabsa_labels = ["O"] + [f"B-{s}" for s in sentiments] + [f"I-{s}" for s in sentiments]
uabsa_label2id = {l: i for i, l in enumerate(uabsa_labels)}

print(f"TASD labels: {len(tasd_labels)} | UABSA labels: {len(uabsa_labels)}")

SPLIT_VAL = "dev"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU available: True
Device: Tesla T4
restaurant_en shape: (3295, 8)
TASD labels: 79 | UABSA labels: 7


In [ ]:
# ============================================================
# BLOCK C: mT5 TASD — requires BLOCK 0 only
# FIXED: label padding masked to -100, use_cache=False set,
# checkpoint save added (needed for ensemble downstream)
# ============================================================
import numpy as np
import os, shutil, torch
from transformers import (AutoTokenizer, MT5ForConditionalGeneration,
                           Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq)
from datasets import Dataset

def build_seq2seq_df(source_df, task="tasd"):
    records = []
    for (sentence, split), group in source_df.groupby(["sentence", "split"]):
        targets = []
        for _, row in group.iterrows():
            aspect = str(row.aspect).strip()
            if aspect.upper() == "NULL" or aspect == "" or pd.isna(row.aspect):
                continue
            if task == "tasd":
                targets.append(f"{aspect} | {row.category} | {row.sentiment}")
            else:
                targets.append(f"{aspect} | {row.sentiment}")
        target_text = " ; ".join(targets) if targets else "none"
        records.append({"sentence": sentence, "split": split, "target": target_text})
    return pd.DataFrame(records)

tasd_seq2seq_df = build_seq2seq_df(restaurant_en, task="tasd")

mt5_tokenizer = AutoTokenizer.from_pretrained("google/mt5-base")

def make_mt5_tokenize_fn(prefix="Extract aspects and sentiments: "):
    def tokenize_fn(examples):
        inputs = [prefix + s for s in examples["sentence"]]
        model_inputs = mt5_tokenizer(inputs, max_length=128, truncation=True, padding="max_length")

        labels = mt5_tokenizer(text_target=examples["target"], max_length=128, truncation=True)
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs
    return tokenize_fn

tasd_mt5_tok_fn = make_mt5_tokenize_fn()

def prep_mt5_dataset(seq2seq_df, tok_fn):
    train_ds = Dataset.from_pandas(seq2seq_df[seq2seq_df.split == "train"][["sentence", "target"]]).map(tok_fn, batched=True)
    val_ds = Dataset.from_pandas(seq2seq_df[seq2seq_df.split == SPLIT_VAL][["sentence", "target"]]).map(tok_fn, batched=True)
    return train_ds, val_ds

tasd_mt5_train, tasd_mt5_val = prep_mt5_dataset(tasd_seq2seq_df, tasd_mt5_tok_fn)
print("TASD mT5 train/val:", len(tasd_mt5_train), len(tasd_mt5_val))


mt5_data_collator = DataCollatorForSeq2Seq(
    tokenizer=mt5_tokenizer, model=None, label_pad_token_id=-100
)


def train_mt5(train_ds, val_ds, out_dir, epochs=15, seed=42):
    torch.manual_seed(seed)
    model = MT5ForConditionalGeneration.from_pretrained("google/mt5-base")
    model.config.use_cache = False

    args = Seq2SeqTrainingArguments(
        output_dir=out_dir,
        eval_strategy="epoch", save_strategy="no",
        learning_rate=3e-4,
        per_device_train_batch_size=2, per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=epochs, weight_decay=0.01, logging_steps=50,
        predict_with_generate=True, generation_max_length=128,
        bf16=True, gradient_checkpointing=True,
        seed=seed,
    )
    trainer = Seq2SeqTrainer(
        model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
        processing_class=mt5_tokenizer, data_collator=mt5_data_collator,
    )
    trainer.train()
    return trainer

def parse_tasd_output(text):
    text = text.strip()
    if text.lower() == "none" or text == "":
        return set()
    triples = set()
    for chunk in text.split(";"):
        parts = [p.strip() for p in chunk.split("|")]
        if len(parts) == 3:
            triples.add((parts[0].lower(), parts[1].lower(), parts[2].lower()))
    return triples

def evaluate_mt5_tasd(trainer, val_ds, tokenizer, num_beams=4):
    predictions = trainer.predict(val_ds, num_beams=num_beams, max_length=128)
    pred_ids = predictions.predictions
    label_ids = predictions.label_ids

    pred_ids = np.where(pred_ids < 0, tokenizer.pad_token_id, pred_ids).astype(np.int64)
    label_ids = np.where(label_ids != -100, label_ids, tokenizer.pad_token_id).astype(np.int64)
    vocab_size = tokenizer.vocab_size + len(tokenizer.get_added_vocab())
    pred_ids = np.clip(pred_ids, 0, vocab_size - 1)
    label_ids = np.clip(label_ids, 0, vocab_size - 1)

    pred_texts = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    gold_texts = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    tp, fp, fn = 0, 0, 0
    for pred_text, gold_text in zip(pred_texts, gold_texts):
        pred_set = parse_tasd_output(pred_text)
        gold_set = parse_tasd_output(gold_text)
        tp += len(pred_set & gold_set)
        fp += len(pred_set - gold_set)
        fn += len(gold_set - pred_set)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return {"f1": f1, "precision": precision, "recall": recall, "tp": tp, "fp": fp, "fn": fn,
            "sample_preds": list(zip(pred_texts[:8], gold_texts[:8]))}

def save_mt5_checkpoint(trainer, tokenizer, local_dir, drive_dir):
    os.makedirs(local_dir, exist_ok=True)
    trainer.save_model(local_dir)
    tokenizer.save_pretrained(local_dir)
    # verify reload before trusting the save
    check = MT5ForConditionalGeneration.from_pretrained(local_dir)
    print(f"Saved + verified locally at {local_dir}")

    # copy to Drive per save-reliability rule (local first, then copytree)
    os.makedirs(os.path.dirname(drive_dir), exist_ok=True)
    shutil.copytree(local_dir, drive_dir, dirs_exist_ok=True)
    # confirm the safetensors file actually landed on Drive, not just the config
    has_weights = any(f.endswith(".safetensors") or f == "pytorch_model.bin"
                       for f in os.listdir(drive_dir))
    print(f"Copied to Drive at {drive_dir} — weights present: {has_weights}")
    return has_weights

# --- RUN TRAINING + EVAL + SAVE ---
tasd_mt5_trainer = train_mt5(tasd_mt5_train, tasd_mt5_val, "/content/mt5_tasd_out", epochs=15)

tasd_mt5_results = evaluate_mt5_tasd(tasd_mt5_trainer, tasd_mt5_val, mt5_tokenizer, num_beams=4)

print("\n=== mT5 TASD RESULTS (15 epochs) ===")
print(f"F1: {tasd_mt5_results['f1']:.4f}")
print(f"Precision: {tasd_mt5_results['precision']:.4f}")
print(f"Recall: {tasd_mt5_results['recall']:.4f}")

save_mt5_checkpoint(
    tasd_mt5_trainer, mt5_tokenizer,
    local_dir="/content/mt5_tasd_final",
    drive_dir="/content/drive/MyDrive/MABSA_Shared_Project/models/mt5_tasd_final",
)

config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/376 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

Map:   0%|          | 0/1263 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Saved + verified locally at /content/mt5_tasd_final
Copied to Drive at /content/drive/MyDrive/MABSA_Shared_Project/models/mt5_tasd_final — weights present: True


True

In [ ]:
# ============================================================
# GRID SEARCH: tune ensemble weights/threshold on DEV split,
# then apply best config to TEST split once (avoids tuning-on-test leakage)
# Run after mT5 + XLM-R are loaded and GPU cleanup has been done.
# ============================================================

import itertools


dev_df = restaurant_en[restaurant_en.split == SPLIT_VAL].copy()

dev_sentences = []
dev_gold_sets = []
for sentence, group in dev_df.groupby("sentence"):
    dev_sentences.append(sentence)
    dev_gold_sets.append(gold_triples_for_sentence(group))
print(f"Dev sentences: {len(dev_sentences)}")


print("Running XLM-R on dev...")
xlmr_dev_spans = xlmr_predict_spans_batch(dev_sentences)

print("Running mT5 on dev...")
mt5_dev_spans = mt5_predict_spans_batch(dev_sentences)


xlmr_weight_grid = [0.6, 0.8, 1.0, 1.2, 1.4]
mt5_weight_grid   = [0.6, 0.8, 1.0, 1.2, 1.4]
threshold_grid    = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
agree_bonus_grid  = [1.0, 1.3, 1.6]

def ensemble_combine_v2(xlmr_spans, mt5_spans, xlmr_weight, mt5_weight, threshold, agree_bonus):
    votes = defaultdict(float)
    for s in xlmr_spans:
        key = (s["aspect"].lower(), s["category"].lower(), s["sentiment"].lower())
        votes[key] += s["conf"] * xlmr_weight
    for s in mt5_spans:
        key = (s["aspect"].lower(), s["category"].lower(), s["sentiment"].lower())
        votes[key] += s["conf"] * mt5_weight

    xlmr_keys = {(s["aspect"].lower(), s["category"].lower(), s["sentiment"].lower()) for s in xlmr_spans}
    mt5_keys = {(s["aspect"].lower(), s["category"].lower(), s["sentiment"].lower()) for s in mt5_spans}
    for key in xlmr_keys & mt5_keys:
        votes[key] *= agree_bonus

    return {key for key, v in votes.items() if v >= threshold}

best_config = None
best_f1 = -1.0
results_log = []

total_combos = len(xlmr_weight_grid) * len(mt5_weight_grid) * len(threshold_grid) * len(agree_bonus_grid)
print(f"\nSearching {total_combos} combos on dev set...")

for xw, mw, th, ab in itertools.product(xlmr_weight_grid, mt5_weight_grid, threshold_grid, agree_bonus_grid):
    pred_sets = [ensemble_combine_v2(x, m, xw, mw, th, ab)
                 for x, m in zip(xlmr_dev_spans, mt5_dev_spans)]
    score = score_triples(pred_sets, dev_gold_sets)
    results_log.append({"xlmr_weight": xw, "mt5_weight": mw, "threshold": th,
                         "agree_bonus": ab, **score})
    if score["f1"] > best_f1:
        best_f1 = score["f1"]
        best_config = {"xlmr_weight": xw, "mt5_weight": mw, "threshold": th, "agree_bonus": ab}

print(f"\nBest dev config: {best_config}")
print(f"Best dev F1: {best_f1:.4f}")


def ensemble_union(xlmr_spans, mt5_spans):
    xlmr_keys = {(s["aspect"].lower(), s["category"].lower(), s["sentiment"].lower()) for s in xlmr_spans}
    mt5_keys = {(s["aspect"].lower(), s["category"].lower(), s["sentiment"].lower()) for s in mt5_spans}
    return xlmr_keys | mt5_keys

union_pred_sets = [ensemble_union(x, m) for x, m in zip(xlmr_dev_spans, mt5_dev_spans)]
union_dev_score = score_triples(union_pred_sets, dev_gold_sets)
print(f"\nSimple union baseline (dev): F1={union_dev_score['f1']:.4f}  "
      f"P={union_dev_score['precision']:.4f}  R={union_dev_score['recall']:.4f}")


with open(f"{DRIVE}/results/ensemble_grid_search_dev.json", "w") as f:
    json.dump({"best_config": best_config, "best_dev_f1": best_f1,
                "union_baseline_dev": union_dev_score,
                "all_results": results_log}, f, indent=2)
print(f"Saved grid search log to {DRIVE}/results/ensemble_grid_search_dev.json")


final_ensemble_pred_sets = [
    ensemble_combine_v2(x, m, best_config["xlmr_weight"], best_config["mt5_weight"],
                         best_config["threshold"], best_config["agree_bonus"])
    for x, m in zip(xlmr_all_spans, mt5_all_spans)
]
final_ensemble_score = score_triples(final_ensemble_pred_sets, gold_sets)

union_test_pred_sets = [ensemble_union(x, m) for x, m in zip(xlmr_all_spans, mt5_all_spans)]
union_test_score = score_triples(union_test_pred_sets, gold_sets)

print("\n=== FINAL TEST SET RESULTS (tuned on dev, applied once) ===")
print(f"XLM-R alone       : F1={xlmr_score['f1']:.4f}  P={xlmr_score['precision']:.4f}  R={xlmr_score['recall']:.4f}")
print(f"mT5 alone         : F1={mt5_score['f1']:.4f}  P={mt5_score['precision']:.4f}  R={mt5_score['recall']:.4f}")
print(f"Ensemble (untuned): F1={ensemble_score['f1']:.4f}  P={ensemble_score['precision']:.4f}  R={ensemble_score['recall']:.4f}")
print(f"Ensemble (tuned)  : F1={final_ensemble_score['f1']:.4f}  P={final_ensemble_score['precision']:.4f}  R={final_ensemble_score['recall']:.4f}")
print(f"Union baseline    : F1={union_test_score['f1']:.4f}  P={union_test_score['precision']:.4f}  R={union_test_score['recall']:.4f}")


final_results = {
    "xlmr_alone": xlmr_score,
    "mt5_alone": mt5_score,
    "ensemble_untuned": ensemble_score,
    "ensemble_tuned": final_ensemble_score,
    "union_baseline": union_test_score,
    "best_config_from_dev": best_config,
    "n_test_sentences": len(test_sentences),
    "n_dev_sentences": len(dev_sentences),
}
with open(f"{DRIVE}/results/ensemble_tasd_final_comparison.json", "w") as f:
    json.dump(final_results, f, indent=2)
print(f"\nSaved final comparison to {DRIVE}/results/ensemble_tasd_final_comparison.json")

Dev sentences: 316
Running XLM-R on dev...


[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Running mT5 on dev...


[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/


Searching 450 combos on dev set...

Best dev config: {'xlmr_weight': 0.6, 'mt5_weight': 0.6, 'threshold': 0.4, 'agree_bonus': 1.0}
Best dev F1: 0.6694

Simple union baseline (dev): F1=0.6479  P=0.6038  R=0.6989
Saved grid search log to /content/drive/MyDrive/MABSA_Shared_Project/results/ensemble_grid_search_dev.json

=== FINAL TEST SET RESULTS (tuned on dev, applied once) ===
XLM-R alone       : F1=0.5159  P=0.6991  R=0.4087
mT5 alone         : F1=0.6509  P=0.6553  R=0.6466
Ensemble (untuned): F1=0.4925  P=0.7948  R=0.3568
Ensemble (tuned)  : F1=0.6645  P=0.6511  R=0.6784
Union baseline    : F1=0.6573  P=0.6143  R=0.7069

Saved final comparison to /content/drive/MyDrive/MABSA_Shared_Project/results/ensemble_tasd_final_comparison.json


In [ ]:
# ============================================================
# SAVE ENSEMBLE RESULTS — consolidated, reload-verified save
# ============================================================

import json, os

DRIVE = "/content/drive/MyDrive/MABSA_Shared_Project"
os.makedirs(f"{DRIVE}/results", exist_ok=True)

ensemble_final_results = {
    "task": "TASD",
    "domain": "restaurant",
    "language": "english",
    "n_dev_sentences": len(dev_sentences),
    "n_test_sentences": len(test_sentences),
    "scores": {
        "xlmr_alone": xlmr_score,
        "mt5_alone": mt5_score,
        "ensemble_untuned": {
            **ensemble_score,
            "config": {"xlmr_weight": 1.0, "mt5_weight": 1.0, "threshold": "0.5*(w1+w2)", "agree_bonus": 1.3}
        },
        "ensemble_tuned": {
            **final_ensemble_score,
            "config": best_config
        },
        "union_baseline": union_test_score,
    },
    "best_config_source": "tuned on dev split via grid search, applied once to test",
    "notes": (
        "Ensemble combines XLM-R (token classification) and mT5 (seq2seq) predictions "
        "at the span level via confidence-weighted voting. Tuned ensemble outperforms "
        "both individual models and the simple union baseline. Weights/threshold selected "
        "on dev, evaluated once on test to avoid tuning-on-test leakage."
    ),
}

save_path = f"{DRIVE}/results/ensemble_tasd_final_results.json"
with open(save_path, "w") as f:
    json.dump(ensemble_final_results, f, indent=2)

with open(save_path) as f:
    check = json.load(f)
print(f"Saved + verified: {save_path}")
print(f"Keys present: {list(check['scores'].keys())}")
print(f"Tuned ensemble F1: {check['scores']['ensemble_tuned']['f1']:.4f}")

consolidated_path = f"{DRIVE}/results/final_consolidated_results.json"
if os.path.exists(consolidated_path):
    with open(consolidated_path) as f:
        all_results = json.load(f)
    all_results["ensemble"] = ensemble_final_results
    with open(consolidated_path, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"Also merged into {consolidated_path} under key 'ensemble'")
else:
    print("final_consolidated_results.json not found yet — skipped merge (ensemble result still saved standalone above)")